In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GATConv
import torch_geometric.transforms as T
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:
# Define the GAT model
# this implementation is credit to pytorch_geometric examples
class GAT(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads):
        super(GAT, self).__init__()
        self.conv1 = GATConv(in_channels, hidden_channels, heads, dropout=0.6)
        self.conv2 = GATConv(hidden_channels * heads, out_channels, heads=1, concat=False, dropout=0.6)

    def forward(self, data):
        h, edge_index = data.x, data.edge_index

        h = F.dropout(h, p=0.6, training=self.training)
        h = F.elu(self.conv1(h, edge_index))
        h = F.dropout(h, p=0.6, training=self.training)
        h = self.conv2(h, edge_index)

        return h

# Load the datasets
pubmed_dataset = Planetoid(root='/tmp/Pubmed', name='Pubmed', transform=T.NormalizeFeatures())
data = pubmed_dataset[0]
data = data.to(device)

In [3]:
h_channels = 64
heads = 8
model = GAT(pubmed_dataset.num_features, h_channels, pubmed_dataset.num_classes, heads)
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
def train(model, data, train_mask, labels):
    model.train()

    optimizer.zero_grad()
    logits = model(data.cuda())
    loss = F.cross_entropy(logits[train_mask], labels[train_mask])
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()

In [4]:
h_channels = 64
heads = 8
model = GAT(pubmed_dataset.num_features, h_channels, pubmed_dataset.num_classes, heads)
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
def train(model, data, train_mask, labels):
    model.train()

    optimizer.zero_grad()
    logits = model(data.cuda())
    loss = F.cross_entropy(logits[train_mask], labels[train_mask])
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()

In [5]:
train(model, data, data.train_mask, data.y)

1.0952547788619995

In [6]:
@torch.no_grad()
def test():
    model.eval()
    out = model(data)
    pred = out.argmax(dim=1)

    acc = (pred[data.test_mask] == data.y[data.test_mask]).sum().item() / data.test_mask.sum().item()
    return acc

for epoch in range(0, 200):
    loss = train(model, data, data.train_mask, data.y)
    acc = test()
    print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Accuracy: {acc:.4f}')

Epoch: 000, Loss: 1.0773, Accuracy: 0.5930
Epoch: 001, Loss: 1.0615, Accuracy: 0.5180
Epoch: 002, Loss: 1.0607, Accuracy: 0.5130
Epoch: 003, Loss: 1.0297, Accuracy: 0.5380
Epoch: 004, Loss: 1.0154, Accuracy: 0.5990
Epoch: 005, Loss: 0.9921, Accuracy: 0.6680
Epoch: 006, Loss: 0.9878, Accuracy: 0.6860
Epoch: 007, Loss: 0.9557, Accuracy: 0.6850
Epoch: 008, Loss: 0.9092, Accuracy: 0.6640
Epoch: 009, Loss: 0.9264, Accuracy: 0.5720
Epoch: 010, Loss: 0.8768, Accuracy: 0.5440
Epoch: 011, Loss: 0.9046, Accuracy: 0.5610
Epoch: 012, Loss: 0.8757, Accuracy: 0.6560
Epoch: 013, Loss: 0.8037, Accuracy: 0.7250
Epoch: 014, Loss: 0.8493, Accuracy: 0.7180
Epoch: 015, Loss: 0.7928, Accuracy: 0.7100
Epoch: 016, Loss: 0.8145, Accuracy: 0.7110
Epoch: 017, Loss: 0.7503, Accuracy: 0.7060
Epoch: 018, Loss: 0.6534, Accuracy: 0.7090
Epoch: 019, Loss: 0.6971, Accuracy: 0.7120
Epoch: 020, Loss: 0.7210, Accuracy: 0.7150
Epoch: 021, Loss: 0.6711, Accuracy: 0.7230
Epoch: 022, Loss: 0.6396, Accuracy: 0.7270
Epoch: 023,

In [7]:
torch.save(model.state_dict(), 'pubmed_gat.pt')